In [1]:
import sys
sys.path.append("../")

import sympy as sp
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_point, compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()

In [2]:
kappa  = Eigen.Scalar("kappa")

# Basis vectors
ci_bar = Eigen.Vector("ci_bar",3)
cj_bar = Eigen.Vector("cj_bar",3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi",12)
qj = Eigen.Vector("qj",12)

$$
C_0 = ||\mathbf{c}_i - \mathbf{c}_j ||_2= \mathbf{0} 
$$

$$
\mathbf{F}_{t} = \begin{bmatrix}
\mathbf{c}_i - \mathbf{c}_j\\
\end{bmatrix}_{3 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{t} = \begin{bmatrix}
\mathbf{J}(\bar{\mathbf{c}}_i) & -\mathbf{J}(\bar{\mathbf{c}}_j) 
\end{bmatrix}_{3 \times 24}
$$

$$
\mathbf{F}_{t} = \mathbf{J}_{t} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [3]:
Jt = sp.Matrix.zeros(3,24)
Jt[0:3, 0:12] = compute_J_point(ci_bar)
Jt[0:3, 12:24] = -compute_J_point(cj_bar)

content = ""
# from ABD q to F
Ft = Jt @ sp.Matrix.vstack(qi, qj)
Cl_Ft = Gen.Closure(
    ci_bar,
    qi,  # Affine Body i
    cj_bar,
    qj,
)  # Affine Body j
# from F Gradient to ABD Gradient
Gt = Eigen.Vector("Gt3", 3)
JtT_Gt = Jt.T @ Gt
Cl_Gt = Gen.Closure(
    Gt,
    ci_bar,
    cj_bar,
)  
# from F Hessian to ABD Hessian
Ht3x3 = Eigen.Matrix("Ht3x3", 3, 3)
JtT_Ht_Jt = Jt.T @ Ht3x3 @ Jt
Cl_Ht = Gen.Closure(
    Ht3x3,
    ci_bar,
    cj_bar,
) 
content += f""" 
// Spherical Joint: C0
// Mapping between ABD qi qj to Ft

{Cl_Ft("Ft", Ft)}
{Cl_Gt("JtT_Gt", JtT_Gt)}
{Cl_Ht("JtT_Ht_Jt", JtT_Ht_Jt)}

"""

Ft = Eigen.Vector("Ft", 3)

Cij = Ft[0:3, 0]


C0 = sqrt(Cij.T * Cij)
Et = 0.5 * kappa * C0**2

Cl_Et = Gen.Closure(kappa, Ft)


dEdFt = VecDiff(Et, Ft)
ddEdFt = VecDiff(dEdFt, Ft)

content += f""" // Spherical Joint Translation Energy and Derivatives

{Cl_Et("Et", Et)}
{Cl_Et("dEdFt", dEdFt)}
{Cl_Et("ddEdFt", ddEdFt)}

// -------------------------------------------------------------------------
"""

In [4]:
path = backend_source_dir("cuda") / "affine_body/constitutions/sym" / "affine_body_spherical_joint.inl"
f = open(path, "w")
f.write(content)
f.close()
print(f"Written to {path}")

Written to /home/ligo/Project/uipc/libuipc/src/backends/cuda/affine_body/constitutions/sym/affine_body_spherical_joint.inl
